# SaaS billing benchmark at scale

Generate a balanced benchmark over the larger `mock_saas_billing` domain, validate and persist it, then evaluate SynBench's multi-agent pipeline.

In [1]:
import json
from collections import Counter

from aieng.syn_data.synbench.display import (  # noqa: E402
    show_actions,
    show_pipeline_messages,
    show_session,
    show_task,
)

from utils.utils import get_root_directory

ROOT = get_root_directory()
DOMAIN_PATH = ROOT / "domains" / "mock_saas_billing"
OUT_DIR = ROOT / "data" / "benchmarks" / "mock_saas_billing"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Implementation root:", ROOT)
print("Domain:", DOMAIN_PATH)
print("Output:", OUT_DIR)

Implementation root: /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/agent_benchmark_generation
Domain: /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/agent_benchmark_generation/domains/mock_saas_billing
Output: /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/agent_benchmark_generation/data/benchmarks/mock_saas_billing


## Step 1 — Load and inspect the scaled domain

Validate the complete domain bundle, then report aggregate sizes instead of printing the full database.

In [2]:
from aieng.syn_data.synbench.domain.loader import load_domain, validate_domain

errors = validate_domain(DOMAIN_PATH)
assert errors == [], f"Domain validation failed: {errors}"
domain = load_domain(DOMAIN_PATH)

print("Domain:", domain.manifest.name)
print("Task types:", list(domain.state_machine["task_types"]))
print("Seed tasks:", len(domain.seed_tasks))
for collection in ("accounts", "subscriptions", "invoices"):
    print(f"{collection}: {len(domain.db[collection])}")

sample_invoice = next(iter(domain.db["invoices"].values()))
print("\nSample invoice:\n", json.dumps(sample_invoice, indent=2))

Domain: mock_saas_billing
Task types: ['inquiry', 'void_invoice', 'refuse_void', 'update_billing_email', 'update_seats']
Seed tasks: 10
accounts: 48
subscriptions: 96
invoices: 192

Sample invoice:
 {
  "invoice_id": "inv_avery_open",
  "account_id": "acct_avery",
  "subscription_id": "sub_avery_1",
  "status": "open",
  "amount_cents": 2898,
  "currency": "USD",
  "due_date": "2026-01-15",
  "billing_address": "100 Market St, Suite 1",
  "line_items": [
    {
      "description": "Starter plan seats",
      "quantity": 3,
      "unit_cents": 966
    }
  ]
}


## Step 2 — Inspect and replay seed tasks

The domain has two hand-verified seeds per task type. Replay one oracle to demonstrate the target database state used for scoring.

In [3]:
from aieng.syn_data.synbench.environment.core import replay_actions
from aieng.syn_data.synbench.environment.hashing import db_hash

for task_type in domain.state_machine["task_types"]:
    ids = [task.id for task in domain.seed_tasks if task.task_type == task_type]
    print(f"{task_type}: {ids}")

seed = domain.seed_tasks[0]
target_env = replay_actions(domain, seed.evaluation_criteria.actions)
print("\nReplayed seed:", seed.id)
print("Target DB hash:", db_hash(target_env.db))

inquiry: ['seed_inquiry_avery', 'seed_inquiry_noah']
void_invoice: ['seed_void_avery', 'seed_void_maya']
refuse_void: ['seed_refuse_avery', 'seed_refuse_luis']
update_billing_email: ['seed_email_noah', 'seed_email_priya']
update_seats: ['seed_seats_avery', 'seed_seats_luis']

Replayed seed: seed_inquiry_avery
Target DB hash: d8a90747cd51adbf7105b338ad1d5d0cff9f87fec41f2a83033c67b77ed7e033


## Step 3 — Generate tasks at scale with balanced constraints

Hold the task type fixed while sampling a fresh eligible invoice/account for each personality. The retry loop avoids duplicate accounts within a task type and skips inactive subscriptions for seat changes. With the defaults below this requests 25 drafts: 5 task types × 5 personalities.

In [ ]:
from aieng.syn_data.synbench.generation.generator import TrajectoryGenerator
from aieng.syn_data.synbench.generation.sampler import ConstraintSampler

TASK_TYPES = list(domain.state_machine["task_types"])
PERSONALITIES = [style["name"] for style in domain.user_simulator["personality_styles"]]
MAX_SAMPLE_ATTEMPTS = 100

sampler = ConstraintSampler(domain, seed=42)
generator = TrajectoryGenerator(domain, seed=42)
drafts = []
generation_rows = []
# generate one task for each task type and personality --> 25 tasks
for task_type in TASK_TYPES:
    used_accounts = set()
    for personality in PERSONALITIES:
        for _ in range(MAX_SAMPLE_ATTEMPTS):
            constraints = sampler.sample(
                task_type=task_type,
                personality_name=personality,
            )
            account_id = constraints.entities["account_id"]
            subscription = constraints.entity_context.get("subscriptions", {})
            if account_id in used_accounts:
                continue
            if task_type == "update_seats" and subscription.get("status") != "active":
                continue
            break
        else:
            raise RuntimeError(
                f"Could not find a distinct eligible account for {task_type}/{personality}"
            )

        used_accounts.add(account_id)
        task_id = f"scale_{task_type}_{personality}"
        draft = generator.generate_one(task_id=task_id, constraints=constraints)
        drafts.append(draft)
        generation_rows.append(
            {
                "task_type": task_type,
                "account_id": account_id,
                "user_name": draft.user_scenario.user_name,
                "personality": personality,
                "invoice_id": constraints.primary_id,
            }
        )

print(f"Generated {len(drafts)} drafts")
for row in generation_rows:
    print(
        f"{row['task_type']:<22} {row['personality']:<14} "
        f"{row['account_id']:<12} {row['invoice_id']}"
    )

Generated 25 drafts
inquiry                rushed         acct_041     inv_041_4
inquiry                anxious        acct_008     inv_008_1
inquiry                rule_breaker   acct_noah    inv_noah_3
inquiry                inconsistent   acct_048     inv_048_2
inquiry                domain_expert  acct_018     inv_018_3
void_invoice           rushed         acct_015     inv_015_3
void_invoice           anxious        acct_014     inv_014_2
void_invoice           rule_breaker   acct_008     inv_008_4
void_invoice           inconsistent   acct_047     inv_047_1
void_invoice           domain_expert  acct_jordan  inv_jordan_4
refuse_void            rushed         acct_045     inv_045_2
refuse_void            anxious        acct_036     inv_036_3
refuse_void            rule_breaker   acct_007     inv_007_4
refuse_void            inconsistent   acct_039     inv_039_4
refuse_void            domain_expert  acct_029     inv_029_2
update_billing_email   rushed         acct_maya    inv_maya_4

## Step 4 — Verify and save

Apply domain rules, exact oracle FSM validation, replay, and action-sequence deduplication. Only passing tasks are persisted.

In [5]:
from aieng.syn_data.synbench.generation.generator import GenerationRun

run = GenerationRun(domain, seed=42)
verified_tasks, rejections = run.verify_drafts(drafts)
task_path = run.write_tasks(verified_tasks, OUT_DIR)

print("Drafts by type:", dict(Counter(task.task_type for task in drafts)))
print("Passing by type:", dict(Counter(task.task_type for task in verified_tasks)))
print("Rejected by type:", dict(Counter(task.task_type for task, _ in rejections)))
print("Saved", len(verified_tasks), "tasks to", task_path)

for task, reasons in rejections:
    print(f"REJECTED {task.id}: {'; '.join(reasons)}")

Drafts by type: {'inquiry': 5, 'void_invoice': 5, 'refuse_void': 5, 'update_billing_email': 5, 'update_seats': 5}
Passing by type: {'inquiry': 5, 'void_invoice': 5, 'refuse_void': 5, 'update_billing_email': 5, 'update_seats': 5}
Rejected by type: {}
Saved 25 tasks to /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/agent_benchmark_generation/data/benchmarks/mock_saas_billing/tasks.json


## Step 5 — Load and re-verify saved tasks

Reload the persisted JSON through the task schema and independently verify every task before evaluation.


**Note: don't forget to load and inspect the domain before running this cell**

In [3]:
from aieng.syn_data.synbench.schemas.tasks import Task
from aieng.syn_data.synbench.verification.pipeline import verify_draft

TASK_PATH = OUT_DIR / "tasks.json"

with TASK_PATH.open() as f:
    saved = json.load(f)

filtered_tasks = []
for item in saved["tasks"]:
    task = Task.model_validate(item)
    verification = verify_draft(domain, task)
    status = "OK" if verification.verification_report.passed else "FAIL"
    print(f"[{status}] {task.id}")
    if verification.verification_report.passed:
        filtered_tasks.append(task)

print(f"Ready to evaluate: {len(filtered_tasks)} tasks")

[OK] scale_inquiry_rushed
[OK] scale_inquiry_anxious
[OK] scale_inquiry_rule_breaker
[OK] scale_inquiry_inconsistent
[OK] scale_inquiry_domain_expert
[OK] scale_void_invoice_rushed
[OK] scale_void_invoice_anxious
[OK] scale_void_invoice_rule_breaker
[OK] scale_void_invoice_inconsistent
[OK] scale_void_invoice_domain_expert
[OK] scale_refuse_void_rushed
[OK] scale_refuse_void_anxious
[OK] scale_refuse_void_rule_breaker
[OK] scale_refuse_void_inconsistent
[OK] scale_refuse_void_domain_expert
[OK] scale_update_billing_email_rushed
[OK] scale_update_billing_email_anxious
[OK] scale_update_billing_email_rule_breaker
[OK] scale_update_billing_email_inconsistent
[OK] scale_update_billing_email_domain_expert
[OK] scale_update_seats_rushed
[OK] scale_update_seats_anxious
[OK] scale_update_seats_rule_breaker
[OK] scale_update_seats_inconsistent
[OK] scale_update_seats_domain_expert
Ready to evaluate: 25 tasks


## Step 6 — Evaluate the multi-agent system

Use the same user simulator → planner → executor → critic pipeline as notebook 4. First inspect one score, then compute pass@1 and mean rewards over all validated tasks.


First inspect the agentic system on one task. Is it evaluated correctly? How can you improve the evaluation?

One Idea: For inquery tasks, make sure the oracle action exists in the agent actions with the expected argument.

In [6]:
from aieng.syn_data.synbench.agents.pipeline import DEFAULT_ROLES, AgentPipeline
from aieng.syn_data.synbench.llm import get_client, get_agent_model
# You can try different LLMs for the agent and see the results
model = get_agent_model()
client = get_client(model)
print("Model:", model)
print("Role order:", DEFAULT_ROLES)

pipeline = AgentPipeline(
    domain,
    max_turns=12,
    max_dialogue_turns=3,
    client=client,
)

first_session = pipeline.run_task(filtered_tasks[0])
first_score = pipeline.score_session(first_session)
print("First task:", filtered_tasks[0].id)
print("Reward:", first_score.reward)
print("DB reward:", first_score.db_reward)
print("Communication reward:", first_score.communicate_reward)

Model: gpt-4o
Role order: ['user_sim', 'planner', 'executor', 'critic']
First task: scale_inquiry_rushed
Reward: 0.0
DB reward: 1.0
Communication reward: 0.0


In [5]:
show_task(first_session.task, title="Task")
show_session(first_session, title="Session (after dialogue)", show_chat=False)
show_pipeline_messages(first_session, title="Pipeline messages")
show_actions(first_session.agent_actions, title="Agent actions")

### Task
**id:** `scale_inquiry_rushed`  
**type:** `inquiry`  
**description:** Look up invoice details and verify current status.

#### User scenario
- **user name:** Finn Evans
- **personality style:** rushed
- **instructions:** Ask the agent to look up invoice inv_041_4 to check its status, amount, and due date.
- **initial message:** Hey, need info on my latest invoice right now. Can you check it?

#### Evaluation criteria
- **reward basis:** DB, COMMUNICATE
- **communicate info:** ['void', '24882', '2027-09-09']
- **oracle actions:**
  - `get_invoice` `{"invoice_id": "inv_041_4"}`

### Session (after dialogue)

| Field | Value |
| --- | --- |
| **domain** | `mock_saas_billing` (`/Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/agent_benchmark_generation/domains/mock_saas_billing`) |
| **task** | `scale_inquiry_rushed` · `inquiry` |
| **env** | ready |
| **messages** | 13 |
| **agent_actions** | 3 |
| **agent_messages** | 3 |
| **role_trace** | ['planner', 'executor', 'critic', 'user_sim', 'planner', 'executor', 'critic', 'user_sim', 'planner', 'executor', 'critic'] |
| **plan** | 1. Request the customer's full name to locate their account and ensure you have the correct account ID.
2. Use the provided invoice ID (`inv_041_4`) to search for the specific invoice details once the account is confirmed.
3. Verify and confirm the invoice details, including amount, status, and due date, with the customer. |

### Agent actions
1. `find_account_id` — `{"name": "Finn Evans"}`
2. `list_invoices` — `{"account_id": "acct_041"}`
3. `get_invoice` — `{"invoice_id": "inv_041_4"}`

### Agent replies

> Certainly! To assist you with your latest invoice, could you please provide your full name? This will help me locate your account.

> Your latest invoice is currently open. Here are the details:
> 
> - **Invoice ID:** inv_041_1
> - **Amount:** $126.00 USD
> - **Status:** Open
> - **Due Date:** July 29, 2027
> - **Billing Address:** 140 Market St, Suite 1
> - **Description:** Pro plan seats
> - **Quantity:** 14 seats
> - **Unit Price:** $9.00 USD per seat
> 
> If you have any further questions or need additional assistance, feel free to ask!

> The invoice with ID `inv_041_4` is currently marked as void. Here are the details:
> 
> - **Amount:** $248.82 USD
> - **Status:** Void
> - **Due Date:** September 9, 2027
> - **Billing Address:** 140 Market St, Suite 4
> - **Description:** Business plan seats
> - **Quantity:** 26 seats
> - **Unit Price:** $9.57 USD per seat
> 
> Since the invoice is void, it is no longer active or due. If you need further assistance or clarification, please let me know.
> 

### Pipeline messages

**role_trace:** `planner → executor → critic → user_sim → planner → executor → critic → user_sim → planner → executor → critic`

**plan (planner):**
```
1. Request the customer's full name to locate their account and ensure you have the correct account ID.
2. Use the provided invoice ID (`inv_041_4`) to search for the specific invoice details once the account is confirmed.
3. Verify and confirm the invoice details, including amount, status, and due date, with the customer.
```

**[0] system**
    You are the billing support agent for this domain.
    task_id: scale_inquiry_rushed

    ## Policy
    # Helio Cloud Billing Support Policy

    You are a billing support agent for Helio Cloud.

    ## Identity and ownership

    1. Customers do not know their `account_id`. Ask for the customer's full name,
       then use `find_account_id`. Never invent or guess an ID.
    2. If no account matches, ask the customer to confirm their full name. Do not
       continue with invoice or subscription tools.
    3. Use the retrieved account ID with `list_invoices` or `list_subscriptions`
       before answering account-level questions or making changes.
    4. Never access or modify an invoice or subscription belonging to another
       account.

    ## Invoices

    5. Look up an invoice and confirm its amount, status, and due date before
       treating an inquiry as complete or attempting to void it.
    6. Only invoices with `open` or `past_due` status may be voided.
    7. A `paid` or already `void` invoice cannot be voided. Explain the status when
       refusing.
    8. Confirm successful voiding to the customer.

    ## Account and subscription changes

    9. Confirm a new billing email with the customer before updating it, and repeat
       the updated email in the final response.
    10. Look up subscriptions before changing seats. Seats may only be changed on
        an `active` subscription, and the new count must be a positive integer.
    11. Repeat the updated seat count in the final response.

    ## Communication

    - Be concise, professional, and explicit about completed or refused changes.
    - Do not expose internal IDs unless the customer provided the corresponding
      invoice identifier.


    Resolve the customer's messages using the provided tools and policy.
    Do not invent goals beyond what the customer says.
    When finished, reply to the customer clearly.

    ## Plan from planner
    1. Request the customer's full name to locate their account and ensure you have the correct account ID.
    2. Use the provided invoice ID (`inv_041_4`) to search for the specific invoice details once the account is confirmed.
    3. Verify and confirm the invoice details, including amount, status, and due date, with the customer.

**[1] user**
    Hey, need info on my latest invoice right now. Can you check it?

**[2] assistant (executor)**
    Certainly! To assist you with your latest invoice, could you please provide your full name? This will help me locate your account.

**[3] user**
    Finn Evans.

**[4] assistant (executor)**
    tool_call `find_account_id`
    {
      "name": "Finn Evans"
    }

**[5] tool**
    {
      "account_id": "acct_041",
      "name": "Finn Evans",
      "email": "finn.evans@example.com",
      "billing_email": "billing+finn.evans@example.com",
      "company": "Evans 41 Software",
      "plan": "pro"
    }

**[6] assistant (executor)**
    tool_call `list_invoices`
    {
      "account_id": "acct_041"
    }

**[7] tool**
    [
      {
        "invoice_id": "inv_041_1",
        "account_id": "acct_041",
        "subscription_id": "sub_041_1",
        "status": "open",
        "amount_cents": 12600,
        "currency": "USD",
        "due_date": "2027-07-29",
        "billing_address": "140 Market St, Suite 1",
        "line_items": [
          {
            "description": "Pro plan seats",
            "quantity": 14,
            "unit_cents": 900
          }
        ]
      },
      {
        "invoice_id": "inv_041_2",
        "account_id": "acct_041",
        "subscription_id": "sub_041_2",
        "status": "paid",
        "amount_cents": 24882,
        "currency": "USD",
        "due_date": "2027-08-12",
        "billing_address": "140 Market St, Suite 2",
        "line_items": [
          {
            "description": "Business plan seats",
            "quantity": 26,
            "unit_cents": 957
          }
        ]
      },
      {
        "invoice_id": "inv_041_3",
        "account_id": "acct_041",
        "subscription_id": "sub_041_1",
        "status": "past_due",
        "amount_cents": 12600,
        "currency": "USD",
        "due_date": "2027-08-26",
        "billing_address": "140 Market St, Suite 3",
        "line_items": [
          {
            "description": "Pro plan seats",
            "quantity": 14,
            "unit_cents": 900
          }
        ]
      },
      {
        "invoice_id": "inv_041_4",
        "account_id": "acct_041",
        "subscription_id": "sub_041_2",
        "status": "void",
        "amount_cents": 24882,
        "currency": "USD",
        "due_date": "2027-09-09",
        "billing_address": "140 Market St, Suite 4",
        "line_items": [
          {
            "description": "Business plan seats",
            "quantity": 26,
            "unit_cents": 957
          }
        ]
      }
    ]

**[8] assistant (executor)**
    Your latest invoice is currently open. Here are the details:

    - **Invoice ID:** inv_041_1
    - **Amount:** $126.00 USD
    - **Status:** Open
    - **Due Date:** July 29, 2027
    - **Billing Address:** 140 Market St, Suite 1
    - **Description:** Pro plan seats
    - **Quantity:** 14 seats
    - **Unit Price:** $9.00 USD per seat

    If you have any further questions or need additional assistance, feel free to ask!

**[9] user**
    That's not the right invoice. Can you check inv_041_4 instead?

**[10] assistant (executor)**
    tool_call `get_invoice`
    {
      "invoice_id": "inv_041_4"
    }

**[11] tool**
    {
      "invoice_id": "inv_041_4",
      "account_id": "acct_041",
      "subscription_id": "sub_041_2",
      "status": "void",
      "amount_cents": 24882,
      "currency": "USD",
      "due_date": "2027-09-09",
      "billing_address": "140 Market St, Suite 4",
      "line_items": [
        {
          "description": "Business plan seats",
          "quantity": 26,
          "unit_cents": 957
        }
      ]
    }

**[12] assistant (executor)**
    The invoice with ID `inv_041_4` is currently marked as void. Here are the details:

    - **Amount:** $248.82 USD
    - **Status:** Void
    - **Due Date:** September 9, 2027
    - **Billing Address:** 140 Market St, Suite 4
    - **Description:** Business plan seats
    - **Quantity:** 26 seats
    - **Unit Price:** $9.57 USD per seat

    Since the invoice is void, it is no longer active or due. If you need further assistance or clarification, please let me know.


### Agent actions
1. `find_account_id` — `{"name": "Finn Evans"}`
2. `list_invoices` — `{"account_id": "acct_041"}`
3. `get_invoice` — `{"invoice_id": "inv_041_4"}`

In [ ]:
from aieng.syn_data.synbench.evaluation.metrics import MetricsCollector

metrics = MetricsCollector()
for task in filtered_tasks:
    score = pipeline.run_and_score_task(task)
    metrics.add(task.id, score)
    print(f"{task.id}: reward={score.reward:.2f}")

summary = metrics.summary()
print("\nBenchmark summary")
print(json.dumps(summary, indent=2))